# Lab 9- Deep Learning Model

This lab is meant to get you started in using Keras to design Deep Neural Networks. The goal here is to simply repeat your previous lab, but with DNNs.

Let's start with reading the data, like before:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

filename="../Lab.7/SUSY.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET", "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
RawNames=["l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi","MET", "MET_phi", "MET_rel", "axial_MET"]
FeatureNames=["M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]

df = pd.read_csv(filename, dtype='float64', names=VarNames)

Now lets define training and test samples. Note that DNNs take very long to train, so for testing purposes we will use only about 10% of the 5 million events in the training/validation sample. Once you get everything working, make the final version of your plots with the full sample. 

Also note that Keras had trouble with the Pandas tensors, so after doing all of the nice manipulation that Pandas enables, we convert the Tensor to a regular numpy tensor.

In [9]:
N_Max=550000
N_Train=500000

Train_Sample=df[:N_Train]
Test_Sample=df[N_Train:N_Max]

X_Train=np.array(Train_Sample[VarNames[1:]])
y_Train=np.array(Train_Sample["signal"])

X_Test=np.array(Test_Sample[VarNames[1:]])
y_Test=np.array(Test_Sample["signal"])


In [ ]:
# extra imports needed for the exercises
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# also need the raw and feature splits for ex2
X_Train_raw = np.array(Train_Sample[RawNames])
X_Test_raw  = np.array(Test_Sample[RawNames])

X_Train_feat = np.array(Train_Sample[FeatureNames])
X_Test_feat  = np.array(Test_Sample[FeatureNames])

# scale everything - makes training much more stable
scaler_all  = StandardScaler().fit(X_Train)
X_Train_sc  = scaler_all.transform(X_Train)
X_Test_sc   = scaler_all.transform(X_Test)

scaler_raw  = StandardScaler().fit(X_Train_raw)
X_Train_raw_sc = scaler_raw.transform(X_Train_raw)
X_Test_raw_sc  = scaler_raw.transform(X_Test_raw)

scaler_feat = StandardScaler().fit(X_Train_feat)
X_Train_feat_sc = scaler_feat.transform(X_Train_feat)
X_Test_feat_sc  = scaler_feat.transform(X_Test_feat)

## Exercise 1

You will need to create several models and make sure they are properly trained. Write a function that takes this history and plots the values versus epoch. For every model that you train in the remainder of this lab, assess:

* Has you model's performance plateaued? If not train for more epochs. 
* Compare the performance on training versus test sample. Are you over training?

In [ ]:
# plots loss and accuracy vs epoch so we can see if the model converged
# and whether train vs val are diverging (overtraining)
def plot_training_history(history, model_name="Model"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history.history['loss']) + 1)

    axes[0].plot(epochs, history.history['loss'], 'b-o', markersize=3, label='train')
    if 'val_loss' in history.history:
        axes[0].plot(epochs, history.history['val_loss'], 'r-o', markersize=3, label='validation')
    axes[0].set_title(f'{model_name} - loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # handle both old and new keras naming
    acc_key     = 'accuracy' if 'accuracy' in history.history else 'acc'
    val_acc_key = 'val_accuracy' if 'val_accuracy' in history.history else 'val_acc'
    axes[1].plot(epochs, history.history[acc_key], 'b-o', markersize=3, label='train')
    if val_acc_key in history.history:
        axes[1].plot(epochs, history.history[val_acc_key], 'r-o', markersize=3, label='validation')
    axes[1].set_title(f'{model_name} - accuracy')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # check for overtraining
    if 'val_loss' in history.history:
        tl = history.history['loss'][-1]
        vl = history.history['val_loss'][-1]
        gap = vl - tl
        if gap > 0.01:
            print(f"{model_name}: train={tl:.4f}, val={vl:.4f}, gap={gap:.4f} -> might be overtraining")
        else:
            print(f"{model_name}: train={tl:.4f}, val={vl:.4f}, gap={gap:.4f} -> looks ok")

    # check if it plateaued
    last5  = history.history['loss'][-5:]
    change = abs(last5[0] - last5[-1])
    if change < 0.001:
        print(f"  loss change over last 5 epochs = {change:.5f} -> plateaued")
    else:
        print(f"  loss change over last 5 epochs = {change:.5f} -> might need more epochs")

## Exercise 2

Following the original paper (see lab 7), make a comparison of the performance (using ROC curves and AUC) between models trained with raw, features, and raw+features data.

In [ ]:
# same 5-layer 300-unit architecture as in the SUSY paper
def build_standard_dnn(input_dim, name="DNN"):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(300, activation='relu'),
        layers.Dense(300, activation='relu'),
        layers.Dense(300, activation='relu'),
        layers.Dense(300, activation='relu'),
        layers.Dense(300, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name=name)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# use early stopping so i don't have to guess the right number of epochs
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("training on raw features...")
model_raw = build_standard_dnn(X_Train_raw_sc.shape[1], name="DNN_Raw")
history_raw = model_raw.fit(X_Train_raw_sc, y_Train, epochs=50, batch_size=1024,
                            validation_split=0.1, callbacks=callbacks, verbose=0)
plot_training_history(history_raw, "DNN - Raw")

print("training on high-level features...")
model_feat = build_standard_dnn(X_Train_feat_sc.shape[1], name="DNN_Feat")
history_feat = model_feat.fit(X_Train_feat_sc, y_Train, epochs=50, batch_size=1024,
                              validation_split=0.1, callbacks=callbacks, verbose=0)
plot_training_history(history_feat, "DNN - Features")

print("training on raw + features...")
model_all = build_standard_dnn(X_Train_sc.shape[1], name="DNN_All")
history_all = model_all.fit(X_Train_sc, y_Train, epochs=50, batch_size=1024,
                            validation_split=0.1, callbacks=callbacks, verbose=0)
plot_training_history(history_all, "DNN - All")

In [ ]:
# ROC comparison for the three feature sets
fig, ax = plt.subplots(figsize=(8, 7))

datasets = [
    (model_raw,  X_Test_raw_sc,  'Raw only',        'steelblue'),
    (model_feat, X_Test_feat_sc, 'High-level only',  'darkorange'),
    (model_all,  X_Test_sc,      'Raw + High-level', 'green'),
]

for model, X_test, label, color in datasets:
    y_score = model.predict(X_test, verbose=0).ravel()
    fpr, tpr, _ = roc_curve(y_Test, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC={roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC - Raw vs Features vs Both')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 3

Design and implement at least 3 different DNN models. Train them and compare performance. You may try different architectures, loss functions, and optimizers to see if there is an effect.

In [ ]:
input_dim = X_Train_sc.shape[1]

# Model A: shallow and wide - just 2 big layers
def build_model_A(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name="ModelA_shallow_wide")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model B: deeper network with dropout to help with overfitting
def build_model_B(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ], name="ModelB_deep_dropout")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Model C: batch norm + L2 reg + RMSprop, wanted to see if a different optimizer helps
def build_model_C(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(300, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dense(300, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dense(300, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ], name="ModelC_batchnorm_l2")
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

# train all 3 with the same settings so comparison is fair
fit_kwargs = dict(x=X_Train_sc, y=y_Train, epochs=50, batch_size=1024,
                  validation_split=0.1, callbacks=callbacks, verbose=0)

print("training model A...")
model_A = build_model_A(input_dim)
history_A = model_A.fit(**fit_kwargs)
plot_training_history(history_A, "Model A - shallow & wide")

print("training model B...")
model_B = build_model_B(input_dim)
history_B = model_B.fit(**fit_kwargs)
plot_training_history(history_B, "Model B - deep + dropout")

print("training model C...")
model_C = build_model_C(input_dim)
history_C = model_C.fit(**fit_kwargs)
plot_training_history(history_C, "Model C - batchnorm + L2 + RMSprop")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

arch_models = [
    (model_A, 'Model A: shallow & wide',           'steelblue'),
    (model_B, 'Model B: deep + dropout',            'darkorange'),
    (model_C, 'Model C: batchnorm + L2 + RMSprop',  'green'),
]

# keep track of best model for ex 4
best_auc   = 0
best_model = None
best_label = ""

for model, label, color in arch_models:
    y_score = model.predict(X_Test_sc, verbose=0).ravel()
    fpr, tpr, _ = roc_curve(y_Test, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC={roc_auc:.4f})')
    if roc_auc > best_auc:
        best_auc   = roc_auc
        best_model = model
        best_label = label

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC - Architecture Comparison')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"best model: {best_label}, AUC={best_auc:.4f}")

## Exercise 4

Repeat exercise 4 from Lab 8, adding your best performing DNN as one of the models.  


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# sklearn is slow on 500k so using 100k like in lab 8
N_sklearn = 100000
X_tr = X_Train_sc[:N_sklearn]
y_tr = y_Train[:N_sklearn]

sklearn_models = {
    "Logistic Regression":     LogisticRegression(max_iter=500),
    "Decision Tree":           DecisionTreeClassifier(max_depth=5),
    "Random Forest":           RandomForestClassifier(n_estimators=100, max_depth=5, n_jobs=-1),
    "AdaBoost":                AdaBoostClassifier(n_estimators=100),
    "Gradient Boosting (BDT)": GradientBoostingClassifier(n_estimators=100, max_depth=3),
}

sklearn_scores = {}
for name, clf in sklearn_models.items():
    print(f"fitting {name}...")
    clf.fit(X_tr, y_tr)
    sklearn_scores[name] = clf.predict_proba(X_Test_sc)[:, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
colors = plt.cm.tab10.colors

for i, (name, y_score) in enumerate(sklearn_scores.items()):
    fpr, tpr, _ = roc_curve(y_Test, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2, linestyle='--',
            label=f'{name} (AUC={roc_auc:.4f})')

# add best DNN from exercise 3
y_score_dnn = best_model.predict(X_Test_sc, verbose=0).ravel()
fpr_dnn, tpr_dnn, _ = roc_curve(y_Test, y_score_dnn)
auc_dnn = auc(fpr_dnn, tpr_dnn)
ax.plot(fpr_dnn, tpr_dnn, color='black', lw=3,
        label=f'Best DNN ({best_label}) (AUC={auc_dnn:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('All classifiers - Lab 8 models + best DNN')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("AUC Summary:")
print("-" * 38)
for name, y_score in sklearn_scores.items():
    fpr, tpr, _ = roc_curve(y_Test, y_score)
    print(f"  {name:<28} {auc(fpr, tpr):.4f}")
print(f"  {'Best DNN':<28} {auc_dnn:.4f}")